In [51]:
# Célula 0 — Instalar bibliotecas necessárias
!pip install scikit-learn

In [52]:
# Celula 1 — Importar bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error,mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression

In [53]:
# Célula 2 — Carregamento do dataset

# Opção 1: CSV
df = pd.read_csv("br_seeg_emissoes_brasil.csv")

# # Visualizar primeiras linhas
# df.head()

# Visualizar informações do dataset
print(f"Shape original: {df.shape}")
# df.shape serve para mostrar o número de linhas e colunas do DataFrame. O resultado é uma tupla (n_linhas, n_colunas).
print(df.info())


Shape original: (454850, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 454850 entries, 0 to 454849
Data columns (total 12 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   ano                  454850 non-null  int64  
 1   nivel_1              454850 non-null  object 
 2   nivel_2              454850 non-null  object 
 3   nivel_3              454850 non-null  object 
 4   nivel_4              454850 non-null  object 
 5   nivel_5              454850 non-null  object 
 6   nivel_6              454850 non-null  object 
 7   tipo_emissao         454850 non-null  object 
 8   gas                  454850 non-null  object 
 9   atividade_economica  453100 non-null  object 
 10  produto              268100 non-null  object 
 11  emissao              406325 non-null  float64
dtypes: float64(1), int64(1), object(10)
memory usage: 41.6+ MB
None


In [54]:
# Célula 3 — Seleção das colunas relevantes

df = df[["ano", "nivel_1", "gas", "emissao"]]

df.head()

print(f"Shape após seleção: {df.shape}")

Shape após seleção: (454850, 4)


In [55]:
# Célula 4 — Filtrar apenas emissões de CO2

df_co2 = df[df["gas"] == "CO2 (t)"].copy() #.copy() é usado para criar uma cópia do DataFrame filtrado, evitando o aviso de "SettingWithCopyWarning" do pandas.
# modifições seguras sem afetar o dataframe original.

print(f"Shape após filtro: {df_co2.shape}")
print(f"\nValores nulos por coluna:")
print(df_co2.isnull().sum())


Shape após filtro: (37550, 4)

Valores nulos por coluna:
ano           0
nivel_1       0
gas           0
emissao    6755
dtype: int64


In [56]:
# Célula 5 — Remoção de valores nulos

# Verificando a quantidade de NaN na coluna "emissao"
total = len(df_co2)
nulos = df_co2["emissao"].isnull().sum()
print(f"Total de registros: {total}")
print(f"Valores nulos na coluna 'emissao': {nulos} ({100*nulos/total:.1f}%)")

'''
    Removendo linhas com valores nulos na coluna "emissao",
    porque ela é nossa variavél alvo (y). Modelos de regressão não conseguem aprender com valores NaN no target.
    
    NaN faz sentido para análise exploratória, mas para treinar modelos de machine learning é obrigatório remove-los.
'''

df_co2 = df_co2.dropna(subset=["emissao"])
print(f"\nApós remoção de NaN: ")
print(f"registros restantes: {len(df_co2)}")
print(f"Valores nulos restantes: {df_co2['emissao'].isnull().sum().sum()}")

Total de registros: 37550
Valores nulos na coluna 'emissao': 6755 (18.0%)

Após remoção de NaN: 
registros restantes: 30795
Valores nulos restantes: 0


In [57]:
# Célula 6 — Aplicação de One-Hot Encoding

'''
    Aplicamos One_Hot Encoding na coluna "nivel_1".
    get_dummies() é uma função do pandas que converte uma coluna categórica em várias colunas binárias (0 ou 1) para cada categoria única.
    drop_first=True é usado para evitar a multicolinearidade, removendo a primeira categoria como referência.
    (se todas dummies são 0, sabemos que é a cotegoria removida).
'''

df_encoded = pd.get_dummies(
    df_co2,
    columns=["nivel_1"],
    drop_first=True, # drop_first=True é usado para evitar a multicolinearidade, removendo a primeira categoria como referência.
    dtype=int
)

print((f"Shape após One-Hot Encoding: {df_encoded.shape}"))
print(f"Colunas: {list(df_encoded.columns)}")
df_encoded.head()

''' 
Nivel_1 possui 5 categorias únicas, então o One-Hot Encoding criou 4 novas colunas (uma a menos por causa do drop_first=True):
    nivel_1_Energia	
    nivel_1_Mudança de Uso da Terra e Floresta	
    nivel_1_Processos Industriais	
    nivel_1_Resíduos
'''


Shape após One-Hot Encoding: (30795, 7)
Colunas: ['ano', 'gas', 'emissao', 'nivel_1_Energia', 'nivel_1_Mudança de Uso da Terra e Floresta', 'nivel_1_Processos Industriais', 'nivel_1_Resíduos ']


' \nNivel_1 possui 5 categorias únicas, então o One-Hot Encoding criou 4 novas colunas (uma a menos por causa do drop_first=True):\n    nivel_1_Energia\t\n    nivel_1_Mudança de Uso da Terra e Floresta\t\n    nivel_1_Processos Industriais\t\n    nivel_1_Resíduos\n'

In [58]:
# Célula 7 — Normalização da variável de resposta (Emissão)

'''
    Normalização Min-Max: transforma os valores para o intervalo [0, 1].
    Fórmula: x_norm = (x - x_min) / (x_max - x_min)
'''

scaler_x = MinMaxScaler() # Para features. features são as colunas de entrada (X).
scaler_y = MinMaxScaler() # Para target (separado, para poder inverter depois)

# Normalizar "ano" e "emissao"
df_encoded[["ano"]] = scaler_x.fit_transform(df_encoded[["ano"]])
df_encoded[["emissao"]] = scaler_y.fit_transform(df_encoded[["emissao"]])
print(f"Estatísticas após normalização:")
df_encoded.describe()


Estatísticas após normalização:


,ano,emissao,nivel_1_Energia,nivel_1_Mudança de Uso da Terra e Floresta,nivel_1_Processos Industriais,nivel_1_Resíduos
count,30795.000000,30795.000000,30795.000000,30795.000000,30795.000000,30795.000000
mean,0.566975,0.130709,0.579640,0.361422,0.047086,0.003247
std,0.278803,0.017723,0.493625,0.480420,0.211825,0.056893
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.367347,0.129799,0.000000,0.000000,0.000000,0.000000
50%,0.591837,0.129799,1.000000,0.000000,0.000000,0.000000
75%,0.795918,0.129844,1.000000,1.000000,0.000000,0.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


# Modelos de Regressão.

Regressão Linear Multipla.  Multiple Linear Regression (MLR)

In [59]:
# Célula 8 - Definição de x (features) e y (target)

'''
    Aqui definimos o que o modelo vai ussar para prever (x),
    e o que queremos prever (y).
    
    x = todas as colunas exceto "emissao" (que é o que vamos prever),
    y = apenas a coluna "emissão".
'''

X = df_encoded.drop(columns=["emissao"])
y = df_encoded["emissao"]

print(f"Features (X): {list(X.columns)}")
print(f"Shape X: {X.shape}")
print(f"Shape Y: {y.shape}")
print(f"\nPrimeiras linhas de X:")
X.head()

Features (X): ['ano', 'gas', 'nivel_1_Energia', 'nivel_1_Mudança de Uso da Terra e Floresta', 'nivel_1_Processos Industriais', 'nivel_1_Resíduos ']
Shape X: (30795, 6)
Shape Y: (30795,)

Primeiras linhas de X:


,ano,gas,nivel_1_Energia,nivel_1_Mudança de Uso da Terra e Floresta,nivel_1_Processos Industriais,nivel_1_Resíduos
14150,0.000000,CO2 (t),0,0,0,0
14151,0.020408,CO2 (t),0,0,0,0
14152,0.040816,CO2 (t),0,0,0,0
14153,0.061224,CO2 (t),0,0,0,0
14154,0.081633,CO2 (t),0,0,0,0


In [60]:
# Célula 9 — Divisão dos dados

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2, # 80% treino, 20% teste
    random_state=42 # Semente para reprodutibilidade
)

print(f"Treino: {X_train.shape[0]} amostras")
print(f"Teste: {X_test.shape[0]} amostras")

Treino: 24636 amostras
Teste: 6159 amostras


In [61]:
# Célula 10 — Função auxiliar para avaliar modelos

def avaliar_modelo(nome, modelo, X_train, y_train, X_test, y_test):
    '''
    Treina o modelo, faz predições e retorna as métricas.
    
    Métricas:
        - MAE (Mean Absolute Error): Erro médio absoluto entre as predições e os valores reais. 
            Quanto menor, melhor.
        - RMSE (Root Mean Squared Error): Raiz do erro quadrático médio entre as predições e os valores reais. 
            Quanto menor, melhor.
        - R² (Coeficiente de Determinação): Proporção da variância da variável alvo que é explicada pelo modelo. 
            Quanto maior, melhor.
                -> R² = 1.0 : modelo perfeito. 0.0 : modelo que não explica nada. Negativo: modelo pior que a média.
    '''
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    print(f"{'='*50}") # 50 caracteres de "=" para separar os resultados
    print(f"{nome}")
    print(f"MAE: {mae:.6f}")
    print(f"RMSE: {rmse:.6f}")
    print(f"R²: {r2:.6f}")
    
    return {"nome" : nome, "MAE": mae, "RMSE": rmse, "R2": r2, "y_pred": y_pred}

In [63]:
# Célula 11 — Modelo 1: Regressão Linear Múltipla (MLR)

mlr = LinearRegression()
res_mlr = avaliar_modelo("Regressão Linear Múltipla", mlr, X_train, X_test, y_train, y_test)

ValueError: could not convert string to float: 'CO2 (t)'